In [ ]:
start-dfs.sh
start-yarn.sh
jps

In [ ]:
NameNode #if this not visible
DataNode
ResourceManager
NodeManager

In [ ]:
hdfs namenode -format   # do this
start-dfs.sh

In [ ]:
mkdir LogAnalysis
cd LogAnalysis

In [ ]:
nano log.txt

In [ ]:
INFO Login success
ERROR Database failed
INFO File uploaded
ERROR Timeout
WARN Disk low

In [ ]:
nano Mapper1.java

In [ ]:
import java.io.IOException;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.Mapper;

public class Mapper1 extends Mapper<Object, Text, Text, IntWritable> {

    private final static IntWritable one = new IntWritable(1);
    private Text logLevel = new Text();

    public void map(Object key, Text value, Context context)
            throws IOException, InterruptedException {

        String line = value.toString();

        if(line.startsWith("ERROR")) {
            logLevel.set("ERROR");
            context.write(logLevel, one);
        }
        else if(line.startsWith("INFO")) {
            logLevel.set("INFO");
            context.write(logLevel, one);
        }
        else if(line.startsWith("WARN")) {
            logLevel.set("WARN");
            context.write(logLevel, one);
        }
    }
}

In [ ]:
nano Reducer1.java

In [ ]:
import java.io.IOException;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.Reducer;

public class Reducer1 extends Reducer<Text, IntWritable, Text, IntWritable> {

    public void reduce(Text key, Iterable<IntWritable> values, Context context)
            throws IOException, InterruptedException {

        int sum = 0;

        for(IntWritable val : values) {
            sum += val.get();
        }

        context.write(key, new IntWritable(sum));
    }
}

In [ ]:
nano Driver.java

In [ ]:
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class Driver {

    public static void main(String[] args) throws Exception {

        Job job = Job.getInstance(new Configuration(), "LogAnalysis");

        job.setJarByClass(Driver.class);

        job.setMapperClass(Mapper1.class);
        job.setReducerClass(Reducer1.class);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(IntWritable.class);

        FileInputFormat.addInputPath(job, new Path("/input"));
        FileOutputFormat.setOutputPath(job, new Path("/output"));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}

In [ ]:
javac -classpath `hadoop classpath` -d . Mapper1.java Reducer1.java Driver.java

In [ ]:
jar -cvf log.jar *.class

In [ ]:
hdfs dfs -mkdir /input

In [ ]:
hdfs dfs -put log.txt /input

In [ ]:
hdfs dfs -ls /input

In [ ]:
hadoop jar log.jar Driver

In [ ]:
hdfs dfs -cat /output/part-r-00000